### Notebook for exploring Self-Conditioned Knots for Probabilistic LiDAR Hazard Fields

In [ ]:
import os, sys
repo_root = os.path.abspath('..')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from vedo import *
settings.default_backend = "k3d"
import numpy as np
from scipy.spatial.transform import Rotation as R
import copy

from matplotlib import pyplot as p

%load_ext autoreload
%autoreload 2
%autosave 180

In [ ]:
# Configure device (GPU if available)
import torch
from torch.utils.data import Dataset, DataLoader

# Reset default device to CPU to avoid DataLoader sampler conflicts
torch.set_default_device(None)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"Device count: {torch.cuda.device_count()}")
print(f"Using device: {device}")


## Generate / load synthetic LiDAR training data (`generate_training_data.py`)

In [ ]:
from pathlib import Path
from generate_training_data import generate_dataset, load_training_dataset

DATA_DIR = Path(repo_root) / "data"
REGENERATE_DATA = True  # Set False to reuse existing scans in ../data/

if REGENERATE_DATA or not (DATA_DIR / "training_scans.npz").exists():
    dataset_info = generate_dataset(
        out_dir=DATA_DIR,
        # Optional overrides (defaults match generate_training_data.py):
        num_poses=10,
        # orbit_radius=0.5,
        # orbit_height=1.4,
        # num_detectors=32,
        # num_azimuth=180,
        # seed=42,
    )
else:
    dataset_info = load_training_dataset(DATA_DIR)
    print(f"Loaded existing dataset from {DATA_DIR.resolve()}")

scene      = dataset_info["scene"]
sensor     = dataset_info["sensor"]
rays_o     = dataset_info["rays_o"]
rays_d     = dataset_info["rays_d"]
hit_ranges = dataset_info["hit_ranges"]
hit_masks  = dataset_info["hit_masks"]
positions  = dataset_info["positions"]
rotations  = dataset_info["rotations"]
N_poses    = len(positions)

print(f"Scene: {scene} | Sensor: {sensor} | Poses: {N_poses} | Total rays: {len(rays_o):,}")


In [ ]:
import numpy as np
from vedo import Plotter, Points
from synthetic_data_generation_utils import PointCloud

# 1. Reproject ray origins + directions out to 3D Cartesian points
valid_hits = (hit_masks > 0.0)
pts_3d = rays_o[valid_hits] + hit_ranges[valid_hits, np.newaxis] * rays_d[valid_hits]
ranges_3d = hit_ranges[valid_hits]

print(f"Reprojected {len(pts_3d):,} 3D points from {len(hit_masks):,} rays "
      f"({valid_hits.mean() * 100:.1f}% hit rate).")

# 2. Package into PointCloud container
cloud_train = PointCloud(
    points=pts_3d,
    probabilities=hit_masks[valid_hits],
    ranges=ranges_3d,
)

# 3. Interactive 3D visualization overlaid with scene geometry
plt_vedo = Plotter(N=1, axes=1, bg=(1, 1, 1), interactive=True)
disp = scene.to_vedo(alpha=0.15) if 'scene' in locals() else []

# Add training point cloud (colored by distance from sensor)
cloud_actor = cloud_train.to_vedo(color_by='range', cmap='plasma', point_size=3.0)
disp.append(cloud_actor)

# Add red markers for sensor orbit positions
if 'positions' in locals():
    disp.append(Points(positions, r=8.0, c='red'))

plt_vedo.show(disp, f"Multi-View Training Data ({len(pts_3d):,} Points from {N_poses} Poses)")


In [ ]:
# Structure into torch dataset for training
from sck_model import LidarRayDataset, GPURayBatcher

# Define the network

In [ ]:
import torch
import torch.nn as nn
from sck_model import PositionalEncoding, DeeperSCKNetwork, SCKNetwork

In [ ]:
from sck_model import exact_continuous_cdf_loss, interpolate_cdf_to_grid, compute_sck_loss, SCKLoss


# Train Loop

### Notes: Self-Conditioned Residual Knot Flow

In true Self-Conditioning (`DeeperSCKNetwork`):
1. **Initial State (Round 0)**: Knots start cleanly and uniformly distributed across $[s_{\min}, s_{\max}]$ ($[0.1\text{m}, 8.0\text{m}]$).
2. **Gap-Bounded Residual Knot Flow**: Endpoints $s_0 = s_{\min}$ and $s_{K-1} = s_{\max}$ stay pinned. Each interior knot predicts a local shift $\delta s_k = \tanh(u_k) \cdot \min(0.45 \min(\Delta s_k^{\text{left}}, \Delta s_k^{\text{right}}), d_{\max})$, guaranteeing strict monotonic ordering without sorting and preventing knot bunching.
3. **Interval Softmax CDF & Ray-Drop Head**: By default (`cdf_mode="softmax"`), interval return probabilities are normalized via softmax with a fixed `0.0` escape bin logit and cumulatively summed, paired with a dedicated PLiNK `ray_drop_head` predicting $p_{\text{hit}}$.
4. **Self-Conditioning Refinement**: Subsequent rounds detach `s_knots` and re-evaluate 3D positional features at the updated knot positions, trained with exact grid-free **Root-CRPS** + ray-drop BCE loss.


In [ ]:
import torch
import logging
from sck_model import train_step as sck_train_step, train_epoch, train_model

# Suppress benign Inductor SM-count warning on mobile GPUs
logging.getLogger("torch._inductor.utils").setLevel(logging.ERROR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
s_max = 15.0  # Covers the ~24m forest clearing (use 8.0 for the legacy 10x10m room)

model = DeeperSCKNetwork(
    num_knots=32,
    s_max=s_max,
    num_rounds=3,
    max_displacement=1.0,
    hidden_dim=256,
    num_hidden_blocks=2,
).to(device)

# Compile model for CUDA kernel fusion (~1.8x speedup on forward pass)
# compiled_model = torch.compile(model) 
compiled_model = model #DEBUG OVERRIDE

optimizer = torch.optim.Adam(compiled_model.parameters(), lr=5e-4, fused=(device.type == 'cuda'))

# Optional: dense grid used strictly for high-resolution plotting/visualization
eval_grid = torch.linspace(0.1, s_max, 256, device=device)

def train_step(batch):
    # Grid-free exact analytic continuous loss (no eval_grid required!)
    return sck_train_step(compiled_model, optimizer, batch, device=device, s_max=s_max)


In [ ]:
import matplotlib.pyplot as plt
import math

# 1. Instantiate Fast GPU Batcher (pure grid-free using exact hit_ranges and hit_masks)
dataset = LidarRayDataset(
    rays_o=rays_o,
    rays_d=rays_d,
    hit_ranges=hit_ranges,
    hit_masks=hit_masks,
    s_max=s_max,
)
dataloader = GPURayBatcher(
    rays_o=rays_o,
    rays_d=rays_d,
    hit_ranges=hit_ranges,
    hit_masks=hit_masks,
    batch_size=1024,  # Peak ~833 MB VRAM on 4GB GPU (leaves >3GB headroom for vedo + HashSCK)
    shuffle=True,
    device=device,
)

# 2. Run Training Loop with Exact Grid-Free Analytic Loss & PLiNK Ray-Drop Head
num_epochs = 20
history = train_model(
    model=compiled_model,
    dataloader=dataloader,
    optimizer=optimizer,
    eval_grid=None,            # None = Exact closed-form grid-free analytic loss!
    num_epochs=num_epochs,
    device=device,
    augment_origin_slide=False, # Vectorized GPU origin sliding on hit_ranges
    l1_weight=0.,              # Ignored in grid-free mode (Root-CRPS used)
    l2_weight=1.0,             # Continuous Ranked Probability Score (CRPS)
    lambda_raydrop=0.1,        # PLiNK ray-drop BCE loss
    s_max=s_max,
)

# 3. Visualize Learned Knot Placements vs Analytic Ground Truth Step Function
model.eval()
with torch.no_grad():
    sample_indices = [0, 50, 150, 300]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    for ax, idx in zip(axes.flat, sample_indices):
        sample = dataset[idx]
        o = sample['ray_o'].unsqueeze(0).to(device)
        d = sample['ray_d'].unsqueeze(0).to(device)
        s_knots, c_knots, _, p_hit = model(o, d)
        s_k = s_knots[0].cpu().numpy()
        c_k = c_knots[0].cpu().numpy()

        # Dense interpolation purely for rendering the predicted continuous curve
        pred_interp = interpolate_cdf_to_grid(s_knots, c_knots, eval_grid)[0].cpu().numpy()
        eval_pts = eval_grid.cpu().numpy()

        # Analytic ground truth step function (PLiNK style)
        if sample['hit_mask'] > 0:
            r_hit = sample['hit_range'].item()
            ax.step([0.1, r_hit, r_hit, s_max], [0.0, 0.0, 1.0, 1.0], 'k--', label=f'GT Step ({r_hit:.2f}m)', linewidth=2, where='post')
            ax.axvline(r_hit, color='green', linestyle=':', alpha=0.6)
        else:
            ax.axhline(0.0, color='gray', linestyle='--', label='GT Sky (Non-Return)', linewidth=2)
            ax.set_title(f'Non-Return Ray (p_hit={p_hit.item():.2f})', fontsize=9, color='purple')

        ax.plot(eval_pts, pred_interp, 'b-', label='Predicted Continuous Spline')
        ax.scatter(s_k, c_k, color='red', s=25, zorder=5, label='Learned Knots')
        ax.set_xlim(0.0, s_max + 0.5)
        ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel('Range (m)')
        ax.set_ylabel('CDF')
        ax.legend(loc='best', fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# Make cloud at novel position

In [ ]:
import torch
import numpy as np
from synthetic_data_generation_utils import PointCloud
from vedo import Plotter
from sck_model import render_novel_point_cloud, generate_point_cloud_from_network

# ==============================================================================
# Execute Inference with PLiNK Ray-Drop Filtering & Visualize
# ==============================================================================

# 1. Generate point cloud with the sensor placed at room center [0, 0, 1.5]
#    Rays with P(non-return) > non_return_threshold are pruned to suppress sky floaters.
pred_cloud = render_novel_point_cloud(
    model=model,
    sensor=sensor,
    sensor_position=[0.0, 0.0, 1.5],
    method='sample',
    # method= 'threshold', #'sample',             # Options: 'sample' (stochastic) or 'threshold' (median/quantile)
    # threshold=0.7,               # CDF threshold if using method='threshold' (0.5 = median return)
    non_return_threshold=0.1,    # Prune ray if likelihood of non-return > 10%
    device=device,
    verbose=True,                # Prints total rays, returned points, and dropped rays
)

# 2. 3D interactive visualization with Vedo (overlaid on scene geometry)
plt_vedo = Plotter(N=1, axes=1, bg=(1, 1, 1), interactive=True)
disp = scene.to_vedo(alpha=0.15)
disp.append(pred_cloud.to_vedo(color_by='range', cmap='plasma', point_size=3.5))
plt_vedo.show(disp, 'Network Reconstructed Point Cloud (with Ray-Drop Filtering)')


In [ ]:
print("Model device:", next(model.parameters()).device)
print(f"Allocated VRAM: {torch.cuda.memory_allocated() / (1024**2):.2f} MB")

---
# Instant-NGP Multi-Resolution Hash-Grid SCK (`hash_sck_model.py`)

Self-contained 2nd implementation replacing the heavy Fourier `H=256` trunk with a **fused Triton 3D Multi-Resolution Hash Grid** (`HashGrid3D`, with automatic fallback to `tinycudann` if installed or pure PyTorch) + lightweight `H=64` decoder:
- **Exact Triton `dL/dtable` & `dL/dx`:** Gradients flow directly from shifted interval midpoints back into knot displacements.
- **Origin-invariant Plücker ray context `(d, m = o x d)`:** Projected once per ray; CDF head defaults to `local_cdf=True` (3D-only readout for cross-ray consistency).
- **Shifted interval-midpoint CDF evaluation:** Interval softmax with `log(delta_s)` physical width scaling + fixed `0.0` escape bin (or `cdf_mode='hazard'`).
- **3D-grounded `ray_drop_head` + sky empty-space carving (`lambda_sky_cdf`)** + zero-sync GPU training loop.


In [ ]:
# 1. Instantiate HashSCKNetwork & High-Throughput GPU Batcher
import torch
from sck_model import GPURayBatcher, LidarRayDataset, interpolate_cdf_to_grid
from hash_sck_model import HashSCKNetwork, train_hash_sck_model

s_max_hash = 15.0
hash_model = HashSCKNetwork(
    num_knots=32,
    s_min=0.1,
    s_max=s_max_hash,
    hidden_dim=64,
    num_hidden_blocks=1,
    num_rounds=2,
    max_displacement=1.0,
    cdf_mode='softmax',      # 'softmax' (width-aware + escape bin) or 'hazard' (Beer-Lambert)
    local_cdf=True,          # True = pure 3D hash readout for cross-ray consistency (Rung 5a)
    n_levels=12,
    n_features=2,
    log2_hashmap_size=16,    # 65,536 entries/level (~6.3 MB VRAM total)
    base_resolution=16,
    max_resolution=512,
    scene_bound=s_max_hash,
    hash_backend='auto',     # Uses tinycudann if installed, else fused Triton, else PyTorch
).to(device)
hash_model = torch.compile(hash_model, dynamic = True)

# Much smaller decoder allows larger batches at ~320 MB peak VRAM!
hash_dataloader = GPURayBatcher(
    rays_o=rays_o,
    rays_d=rays_d,
    hit_ranges=hit_ranges,
    hit_masks=hit_masks,
    batch_size= 4096, #default 2048
    shuffle=True,
    device=device,
)
hash_dataset = LidarRayDataset(
    rays_o=rays_o,
    rays_d=rays_d,
    hit_ranges=hit_ranges,
    hit_masks=hit_masks,
    s_max=s_max_hash,
)


In [ ]:
hash_model.reset_weights(seed=42)

In [ ]:
# 2. Train HashSCKNetwork & Plot Learned Knots vs Ground Truth
hash_history = train_hash_sck_model(
    model=hash_model,
    dataloader=hash_dataloader,
    num_epochs=20,
    lr_mlp=1e-3,
    lr_hash=1e-2,              # Instant-NGP best practice: higher LR for hash table
    use_scheduler=True,
    augment_origin_slide=False, # Set True to augment near-field/multi-range views on GPU
    lambda_raydrop=0.1,
    lambda_sky_cdf=0.1,        # Supervises non-returning rays toward C(t)=0 (3D empty-space carving)
    s_max=s_max_hash,
    device=device,
)

total_time = sum(hash_history['epoch_times'])
print(f"Total training time: {total_time:.2f}s (avg {total_time / len(hash_history['epoch_times']):.2f}s/epoch)")

# Visualize Learned Knot Placements vs Analytic Ground Truth Step Function
hash_eval_grid = torch.linspace(0.1, s_max_hash, 256, device=device)
hash_model.eval()
with torch.no_grad():
    sample_indices = [0, 50, 150, 300]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    for ax, idx in zip(axes.flat, sample_indices):
        sample = hash_dataset[idx]
        o = sample['ray_o'].unsqueeze(0).to(device)
        d = sample['ray_d'].unsqueeze(0).to(device)
        s_knots, c_knots, _, p_hit = hash_model(o, d)
        s_k = s_knots[0].cpu().numpy()
        c_k = c_knots[0].cpu().numpy()

        pred_interp = interpolate_cdf_to_grid(s_knots, c_knots, hash_eval_grid)[0].cpu().numpy()
        eval_pts = hash_eval_grid.cpu().numpy()

        if sample['hit_mask'] > 0:
            r_hit = sample['hit_range'].item()
            ax.step([0.1, r_hit, r_hit, s_max_hash], [0.0, 0.0, 1.0, 1.0], 'k--', label=f'GT Step ({r_hit:.2f}m)', linewidth=2, where='post')
            ax.axvline(r_hit, color='green', linestyle=':', alpha=0.6)
            ax.set_title(f'Hit Ray (p_hit={p_hit.item():.2f})', fontsize=9)
        else:
            ax.axhline(0.0, color='gray', linestyle='--', label='GT Sky (Non-Return)', linewidth=2)
            ax.set_title(f'Non-Return Ray (p_hit={p_hit.item():.2f})', fontsize=9, color='purple')

        ax.plot(eval_pts, pred_interp, 'b-', label='HashSCK Spline')
        ax.scatter(s_k, c_k, color='red', s=25, zorder=5, label='Learned Knots')
        ax.set_xlim(0.0, s_max_hash + 0.5)
        ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel('Range (m)')
        ax.set_ylabel('CDF')
        ax.legend(loc='best', fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
# 3. Render Novel-View Point Cloud from HashSCKNetwork & Visualize in Vedo
from vedo import Plotter
from sck_model import render_novel_point_cloud

hash_pred_cloud = render_novel_point_cloud(
    model=hash_model,
    sensor=sensor,
    sensor_position=[0.0, 0.0, 1.5],
    method='sample',
    # method= 'threshold',             # Options: 'sample' (stochastic) or 'threshold' (median/quantile)
    # threshold=0.001,               # CDF threshold if using method='threshold' (0.5 = median return)
    non_return_threshold=0.1,   # Prune ray if likelihood of non-return > 10%
    batch_size=2048,
    device=device,
    verbose=True,
)

plt_vedo_hash = Plotter(N=1, axes=1, bg=(1, 1, 1), interactive=True)
disp_hash = scene.to_vedo(alpha=0.15)
disp_hash.append(hash_pred_cloud.to_vedo(color_by='range', cmap='plasma', point_size=3.5))
plt_vedo_hash.show(disp_hash, 'HashSCKNetwork Reconstructed Point Cloud (Instant-NGP 3D Hash Grid)')
